# 00 — Real-rover validation (Week 5 artifact)

This notebook is the **human-facing wrapper** around the validation modules under `roverdevkit.validation`. All logic lives in the Python package so the checks are also enforced as CI gates in `tests/test_rover_comparison.py` and `tests/test_cross_scenario.py`.

## Goals (project_plan.md §6 W5)
1. Run the end-to-end mission evaluator on Pragyan, Yutu-2, and Sojourner using published design parameters and mission scenarios.
2. Compare predicted traverse, peak solar power, and thermal survival against published data.
3. Cross-check evaluator behaviour across canonical scenarios (ranking tests) and single-variable perturbations (sensitivity tests).

## What counts as “validation” here
The traverse simulator produces a *design-space upper bound* on range: given the schema's floor on `drive_duty_cycle` (≥ 0.1), sim predictions are 3–8× higher than published mission totals because real missions interleave long science and thermal-wait windows that our lumped model does not capture. The Week-5 acceptance criteria (below) are phrased around *feasibility* + *thermal survival match* + *peak solar in band* rather than tight ratio-of-predicted-to-published, for exactly this reason. See the docstring of `roverdevkit.validation.rover_comparison` for the full acceptance criteria.

In [1]:
from roverdevkit.validation.rover_comparison import (
    compare_all,
    format_report,
)
from roverdevkit.validation.rover_registry import load_truth_table, registry

## 1. Registry contents

Each entry bundles a `DesignVector` (with imputation notes), a `MissionScenario` (YAML-configured), a gravity constant, a thermal architecture, and rover-specific panel parameters. Imputation notes document every field that was not directly published.

In [2]:
for entry in registry():
    print(f"--- {entry.rover_name} ---")
    print(f"  scenario: {entry.scenario.name}  (lat {entry.scenario.latitude_deg:+.1f} deg, {entry.scenario.mission_duration_earth_days:.0f} d, {entry.scenario.traverse_distance_m:.0f} m cap)")
    print(f"  gravity: {entry.gravity_m_per_s2:.3f} m/s^2")
    print(f"  chassis: {entry.design.chassis_mass_kg:.1f} kg, solar: {entry.design.solar_area_m2:.2f} m^2, battery: {entry.design.battery_capacity_wh:.0f} Wh")
    print(f"  panel_eff: {entry.panel_efficiency:.2f}, dust_factor: {entry.panel_dust_factor:.2f}")
    print(f"  thermal: A={entry.thermal_architecture.surface_area_m2:.2f} m^2, RHU={entry.thermal_architecture.rhu_power_w:.1f} W, hib={entry.thermal_architecture.hibernation_power_w:.1f} W")
    print(f"  imputations: {entry.imputation_notes}")
    print()

--- Pragyan ---
  scenario: chandrayaan3_pragyan  (lat -69.4 deg, 14 d, 500 m cap)
  gravity: 1.625 m/s^2
  chassis: 10.0 kg, solar: 0.50 m^2, battery: 60 Wh
  panel_eff: 0.22, dust_factor: 0.85
  thermal: A=0.25 m^2, RHU=0.0 W, hib=2.0 W
  imputations: wheel_width, wheelbase, grouser_height/count, chassis_mass, nominal_speed_mps, drive_duty_cycle imputed from class heritage and published ops. avionics_power set to 20 W (design-space floor for a 26 kg rover).

--- Yutu-2 ---
  scenario: change4_yutu2_per_lunar_day  (lat +45.5 deg, 5 d, 200 m cap)
  gravity: 1.625 m/s^2
  chassis: 35.0 kg, solar: 1.30 m^2, battery: 130 Wh
  panel_eff: 0.20, dust_factor: 0.55
  thermal: A=0.10 m^2, RHU=15.0 W, hib=5.0 W
  imputations: Yutu-2 is out-of-class (135 kg vs 5-50 kg design space); chassis_mass held at 35 kg ceiling. wheelbase, grouser specs, drive_duty_cycle imputed from published images and the per-lunar-day ~25 m drive distance target.

--- Sojourner ---
  scenario: mpf_sojourner_ares_vallis 

## 2. Published truth table

Digitized from mission publications with explicit low/high bands. Citations live in the `citation` column of `data/published_traverse_data.csv`.

In [3]:
for row in load_truth_table():
    print(f"{row.rover_name}: traverse {row.traverse_m_published:.0f} m " 
          f"(band {row.traverse_m_low:.0f}-{row.traverse_m_high:.0f}), "
          f"peak solar {row.peak_solar_power_w_published:.0f} W "
          f"(band {row.peak_solar_power_w_low:.0f}-{row.peak_solar_power_w_high:.0f}), "
          f"thermal survival = {row.thermal_survival_published}")
    print(f"  citation: {row.citation}")
    print(f"  note: {row.notes}")
    print()

Pragyan: traverse 101 m (band 80-140), peak solar 50 W (band 40-70), thermal survival = False
  citation: ISRO Chandrayaan-3 mission updates (Aug-Sep 2023); Nature SR 14:24178 (2024)
  note: Traverse over Lunar Day 1 only. Rover did NOT survive lunar night (no RHUs); thermal_survival_published=false matches the sim's full-mission hot+cold steady-state check. traverse_m_published is the in-mission total.

Yutu-2: traverse 25 m (band 10-60), peak solar 135 W (band 110-160), thermal survival = True
  citation: Di et al. 2020 Icarus; Ding et al. 2022 Acta Astronautica; CNSA dispatches
  note: Per-lunar-day drive distance, first ~2 years. Yutu-2 carries Pu-238 RHUs for lunar-night survival (surviving 60+ lunar days as of 2025); thermal_survival_published=true conditional on registry's RHU-carrying architecture.

Sojourner: traverse 100 m (band 80-120), peak solar 16 W (band 13-18), thermal survival = True
  citation: Wilcox & Nguyen 1998 IEEE Aerospace; Matijevic et al. 1997 Science 278:176

## 3. Run the evaluator on every rover and score

`compare_all` runs `evaluate()` on each registry entry and applies the Week-5 acceptance criteria. The resulting `ComparisonSummary` is what `tests/test_rover_comparison.py` asserts on.

In [4]:
summary = compare_all()
print(format_report(summary))

Rover      range_pred   range_pub  range_ratio  peak_solar_pred  peak_solar_pub  thermal  motor  PASS?
---------------------------------------------------------------------------------------------------------
Pragyan         500 m       101 m       4.93x          44.8 W          50.0 W  match   ok     PASS
Yutu-2          200 m        25 m       8.00x         136.4 W         135.0 W  match   ok     PASS
Sojourner       150 m       100 m       1.50x          16.6 W          16.0 W  match   ok     PASS
---------------------------------------------------------------------------------------------------------
Pass rate: 3/3


## 4. Per-rover acceptance-criteria breakdown

A more verbose view: each criterion, one line per rover.

In [5]:
for r in summary.results:
    print(f"--- {r.rover_name} --- {'PASS' if r.passes else 'FAIL'}")
    print(f"  range_feasible          = {r.range_feasible}  (predicted {r.range_m_predicted:.0f} m vs low bound {r.truth.traverse_m_low:.0f} m)")
    print(f"  range_below_sanity_ceil = {r.range_below_sanity_ceiling}")
    print(f"  thermal_matches         = {r.thermal_matches}  (predicted {r.metrics.thermal_survival}, published {r.truth.thermal_survival_published})")
    print(f"  motor_and_traversal_ok  = {r.motor_and_traversal_ok}")
    print(f"  peak_solar_in_band      = {r.peak_solar_in_band}  (predicted {r.peak_solar_power_w_predicted:.1f} W, band {r.truth.peak_solar_power_w_low:.0f}-{r.truth.peak_solar_power_w_high:.0f} W)")
    print()

--- Pragyan --- PASS
  range_feasible          = True  (predicted 500 m vs low bound 80 m)
  range_below_sanity_ceil = True
  thermal_matches         = True  (predicted False, published False)
  motor_and_traversal_ok  = True
  peak_solar_in_band      = True  (predicted 44.8 W, band 40-70 W)

--- Yutu-2 --- PASS
  range_feasible          = True  (predicted 200 m vs low bound 10 m)
  range_below_sanity_ceil = True
  thermal_matches         = True  (predicted True, published True)
  motor_and_traversal_ok  = True
  peak_solar_in_band      = True  (predicted 136.4 W, band 110-160 W)

--- Sojourner --- PASS
  range_feasible          = True  (predicted 150 m vs low bound 80 m)
  range_below_sanity_ceil = True
  thermal_matches         = True  (predicted True, published True)
  motor_and_traversal_ok  = True
  peak_solar_in_band      = True  (predicted 16.6 W, band 13-18 W)



## 5. Cross-scenario ranking

Three hand-crafted design archetypes (`large_traverser`, `polar_survivor`, `slope_climber`) evaluated on each canonical scenario. We check that the slope-specialty archetype wins slope capability, and that range is dominated by the fast archetype in benign terrain.

In [6]:
from roverdevkit.validation.cross_scenario import rank_archetypes

for scen_name, rank in rank_archetypes().items():
    print(f"--- {scen_name} ---")
    print(f"  range winner:             {rank.range_winner}")
    print(f"  slope capability winner:  {rank.slope_capability_winner}")
    print(f"  energy margin winner:     {rank.energy_margin_winner}")
    for name, m in rank.per_archetype.items():
        print(f"    {name:16s}: range={m.range_km*1000:6.0f} m  slope_cap={m.slope_capability_deg:4.1f} deg  margin={m.energy_margin_pct:5.1f}%  mass={m.total_mass_kg:5.1f} kg")
    print()

--- equatorial_mare_traverse ---
  range winner:             large_traverser
  slope capability winner:  slope_climber
  energy margin winner:     large_traverser
    large_traverser : range=  5000 m  slope_cap=20.9 deg  margin=100.0%  mass= 46.1 kg
    polar_survivor  : range=  4838 m  slope_cap=16.2 deg  margin=100.0%  mass= 22.8 kg
    slope_climber   : range=  5000 m  slope_cap=22.9 deg  margin=100.0%  mass= 54.1 kg

--- polar_prospecting ---
  range winner:             large_traverser
  slope capability winner:  slope_climber
  energy margin winner:     large_traverser
    large_traverser : range=  2000 m  slope_cap=20.9 deg  margin=  0.0%  mass= 46.1 kg
    polar_survivor  : range=  2000 m  slope_cap=16.2 deg  margin=  0.0%  mass= 22.8 kg
    slope_climber   : range=  2000 m  slope_cap=22.9 deg  margin=  0.0%  mass= 54.1 kg

--- highland_slope_capability ---
  range winner:             large_traverser
  slope capability winner:  slope_climber
  energy margin winner:     large_tra

## 6. One-at-a-time sensitivity

Bump each design variable in turn around a baseline and look at the sign of the change in each mission metric. The CI tests enforce the expected direction.

In [7]:
from roverdevkit.validation.cross_scenario import one_at_a_time_sensitivity

print(f"{'variable':25s} {'d_range_km':>12s} {'d_margin_pct':>14s} {'d_slope_deg':>12s} {'d_mass_kg':>10s}")
for entry in one_at_a_time_sensitivity():
    print(f"{entry.variable:25s} {entry.delta_range_km:+12.3f} {entry.delta_energy_margin_pct:+14.2f} {entry.delta_slope_capability_deg:+12.2f} {entry.delta_total_mass_kg:+10.3f}")

variable                    d_range_km   d_margin_pct  d_slope_deg  d_mass_kg


solar_area_m2                   +0.000          +0.00        +0.05     +1.060
battery_capacity_wh             +0.000          +0.00        +0.06     +1.178
avionics_power_w                +0.000         -12.43        +0.04     +0.707
chassis_mass_kg                 +0.000          +0.00        +0.45    +11.306
wheel_radius_m                  +0.000          +0.00        +3.25     +1.730
nominal_speed_mps               +5.443          +0.00        +0.00     +0.000
drive_duty_cycle                +4.838          +0.00        +0.00     +0.000


## 7. Known limitations of the Week-5 harness

Documented honestly so these are **in-plan** rather than undiscovered blindspots:

1. **Range predictions are upper bounds.** The schema's floor on `drive_duty_cycle` is 0.1; real Pragyan, Yutu-2, and Sojourner ran at effective duty cycles of 0.01–0.02. Predicted range is 3–8× published for this structural reason. We accept this and verify feasibility (predicted ≥ published floor) rather than tight ratios.

2. **Analytical terramechanics floor.** The Bekker-Wong path underestimates DP on loose soil (∼20–40 %) relative to what Ding 2011 testbeds show. Week 7 will add an SCM correction to close this gap; Week 5 acceptance criteria are set loose enough to tolerate the current under-prediction.

3. **Yutu-2 is out-of-class.** At 135 kg, Yutu-2 is outside the 5–50 kg micro-rover design space. Its `chassis_mass_kg` is held at the schema ceiling (35 kg); predicted mass under-calls its real mass proportionally. Documented in its imputation note.

4. **Thermal model is single-node.** Real rovers use MLI + variable-emittance louvers + heat pipes. We represent MLI via effective-radiating-area and low absorptivity. Yutu-2's RHU power is tuned to a plausible effective value rather than the actual RHU wattage.

5. **Sojourner on Mars.** Gravity and irradiance are correctly scaled, but the lunar-night-tuned thermal sink temp is a poor match for Mars. We override `sink_temp_lunar_night_k` for that entry but this is more of a wrapper sanity check than strict physical fidelity.

6. **Constant-slope traverse.** `traverse_sim.py` treats `scenario.max_slope_deg` as a fixed slope for the whole run. We configured Pragyan/Yutu-2/Sojourner scenarios at typical-ops slope (5°), not worst-case (12–15°), to match how the real rovers drove. A proper terrain-path model is v2 work.